In [ ]:
import requests
import pandas as pd
from google.colab import files

# Configuración
API_KEY = ""
VIDEO_ID = "OXT8xdqcAoU"
MAX_COMENTARIOS = 500

comentarios = []
pagina = None

while len(comentarios) < MAX_COMENTARIOS:
    parametros = {
        "key": API_KEY,
        "part": "snippet",
        "videoId": VIDEO_ID,
        "maxResults": min(100, MAX_COMENTARIOS - len(comentarios)),
        "textFormat": "plainText",
        "order": "time"
    }

    if pagina:
        parametros["pageToken"] = pagina

    respuesta = requests.get(
        "https://www.googleapis.com/youtube/v3/commentThreads",
        params=parametros,
        timeout=30
    )

    datos = respuesta.json()

    if respuesta.status_code != 200:
        print("Error:", datos.get("error", {}).get("message"))
        break

    for elemento in datos.get("items", []):
        comentario = elemento["snippet"]["topLevelComment"]["snippet"]

        comentarios.append({
            "texto": comentario["textDisplay"],
            "fecha": comentario["publishedAt"],
            "likes": comentario["likeCount"]
        })

    pagina = datos.get("nextPageToken")

    if not pagina:
        break

# Guardar en Colab y descargar
if comentarios:
    df = pd.DataFrame(comentarios)

    archivo = "/content/comentarios_youtube.csv"
    df.to_csv(archivo, index=False, encoding="utf-8-sig")

    print(f"Comentarios guardados: {len(df)}")
    display(df.head())

    files.download(archivo)
else:
    print("No se descargaron comentarios. Revisa la clave y el video.")

Comentarios guardados: 262


,texto,fecha,likes
0,I used to be a substitute teacher which was bo...,2026-08-23T03:27:44Z,0
1,The gig economy exists because people can't pa...,2026-03-25T17:26:31Z,0
2,annoying loud background music. Why can't you ...,2025-08-16T09:55:44Z,0
3,imagine if they had included mabel app researc...,2025-06-11T05:41:49Z,0
4,Freelancer,2025-04-10T02:21:05Z,0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

# Descargar las stopwords si no están disponibles
nltk.download('stopwords', quiet=True)

# Obtener lista de conectores/stopwords en inglés y español
stop_words = set(stopwords.words('english')).union(set(stopwords.words('spanish')))

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""

    # Convertir a minúsculas
    texto = texto.lower()

    # Eliminar URLs/enlaces
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)

    # Eliminar caracteres especiales y números (manteniendo letras y espacios)
    texto = re.sub(r'[^a-záéíóúüñÁÉÍÓÚÜÑ\s]', '', texto)

    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Filtrar conectores (stopwords)
    palabras = texto.split()
    palabras_filtradas = [palabra for palabra in palabras if palabra not in stop_words]

    return " ".join(palabras_filtradas)

# Crear una copia para no alterar el DataFrame original
df_limpio = df.copy()

# Aplicar la función de limpieza
df_limpio['texto_limpio'] = df_limpio['texto'].apply(limpiar_texto)

# Filtrar filas que hayan quedado vacías después de la limpieza
df_limpio = df_limpio[df_limpio['texto_limpio'] != ""]

# Mostrar el resultado
print(f"Comentarios originales: {len(df)}")
print(f"Comentarios tras la limpieza: {len(df_limpio)}")
display(df_limpio[['texto', 'texto_limpio']].head(10))